# ClaimGuard — session 3: the last missing pieces (about 20 min)

Uses the verifier trained in session 2 (no new training):
1. zero-shot NLI on Climate-FEVER and on the contrast set (the comparison was missing);
2. per-example predictions on Climate-FEVER, to complete the error analysis above 100 errors;
3. a second INT8 recipe (feed-forward layers only), because quantizing every layer broke the verifier.

Inputs: **claimguard-code** (upload the NEW `claimguard.zip` as a new version) **and** the output of the
session-2 notebook (*Add Input → Your Work → Notebooks →* the session-2 notebook). GPU T4 x2, Internet on.

## 0. Setup

In [ ]:
import glob, json, os, shutil, subprocess, time
from pathlib import Path

# find the project inside the uploaded dataset (Kaggle unzips it, the folder depth can vary)
candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
              if "claimguard" in Path(p).read_text()]
assert candidates, "claimguard.zip not found: add it as an input dataset (see the first cell)"
SRC = candidates[0]
WORK = Path("/kaggle/working/claimguard")
if not WORK.exists():
    shutil.copytree(SRC, WORK)   # the input folder is read-only, so work on a copy
os.chdir(WORK)
print("project copied from", SRC)

In [ ]:
!pip install -q -r requirements-dev.txt && pip install -q -e . && pip install -q bitsandbytes accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: set the accelerator to GPU T4 x2")

In [ ]:
# run a shell command, log how long it took and if it worked, and NEVER stop the notebook
STATUS_FILE = WORK / "results" / "run_status.json"
STATUS_FILE.parent.mkdir(exist_ok=True)
status = json.loads(STATUS_FILE.read_text()) if STATUS_FILE.exists() else []

def run(name, cmd):
    print(f"\n===== {name} =====\n$ {cmd}", flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=WORK, text=True, capture_output=True)
    minutes = round((time.time() - start) / 60, 1)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-3000:])
    status.append({"step": name, "ok": proc.returncode == 0, "minutes": minutes,
                   "error": None if proc.returncode == 0 else proc.stderr[-1500:]})
    STATUS_FILE.write_text(json.dumps(status, indent=2))
    print(f"-> {'OK' if proc.returncode == 0 else 'FAILED'} in {minutes} min", flush=True)
    return proc.returncode == 0

In [ ]:
# Hugging Face login from the Kaggle secret (only LLM-AggreFact needs it, the rest works without)
HF_OK = False
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    HF_OK = True
    print("Hugging Face login OK")
except Exception as err:
    print("no HF_TOKEN secret, LLM-AggreFact will be skipped:", type(err).__name__)

In [ ]:
# bring back the verifier trained in session 2 (verifier.zip = the models/ folder)
import zipfile
zips = glob.glob("/kaggle/input/**/verifier.zip", recursive=True)
dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/verifier/config.json", recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK / "models")
elif dirs:  # Kaggle sometimes shows the zip already unpacked
    shutil.copytree(dirs[0], WORK / "models" / "verifier", dirs_exist_ok=True)
    calib = glob.glob("/kaggle/input/**/calibration.json", recursive=True)
    if calib:
        shutil.copy(calib[0], WORK / "models" / "calibration.json")
assert (WORK / "models/verifier/config.json").exists(), "session-2 output not found: add it as an input"
print(sorted(p.name for p in (WORK / "models").iterdir()))

## 1. Data, then the checks

In [ ]:
run("data", "make data")
ZS = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
# same command as in session 2: the numbers must be identical (inference is deterministic)
run("climate fine-tuned (+ predictions)", "python scripts/evaluate.py given --dataset climate --config configs/default.yaml")
run("climate zero-shot", f"python scripts/evaluate.py given --dataset climate --config configs/default.yaml --verifier-model {ZS} --tag zeroshot")
run("contrast zero-shot", f"python scripts/evaluate.py given --dataset contrast --config configs/default.yaml --verifier-model {ZS} --tag zeroshot")
run("contrast fine-tuned (+ predictions)", "python scripts/evaluate.py given --dataset contrast --config configs/default.yaml")
run("efficiency: FP32, INT8 all layers, INT8 feed-forward only", "python scripts/evaluate.py efficiency --config configs/default.yaml")

In [ ]:
shutil.make_archive("/kaggle/working/results_session3", "zip", WORK / "results")
print("\n".join(f"{'OK ' if s['ok'] else 'FAIL'} {s['minutes']:>6} min  {s['step']}" for s in status))
print(Path("results/given_climate.json").read_text()[:400])